In [ ]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from tqdm.auto import tqdm
from google.oauth2 import service_account
import numpy as np

# 1. 신분증(JSON 키) 경로 지정
KEY_PATH = '../google_key.json'

# 2. 인증 객체 생성
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# 3. 프로젝트 ID 설정
project_id = 'gdelt-analysis-494301'

# 4. 데이터 불러오기
query = "SELECT SQLDATE FROM `gdelt-bq.full.events` LIMIT 5"
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("인증 성공! 데이터를 가져왔습니다.")

Downloading: 100%|██████████|
인증 성공! 데이터를 가져왔습니다.


In [2]:
# 1. 고위험군 CAMEO 코드 리스트
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE, 
    EventCode,
    GoldsteinScale, 
    NumMentions, 
    AvgTone,
    ActionGeo_Type,
    ActionGeo_Lat, 
    ActionGeo_Long, 
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1                 -- 핵심 사건만 필터링
  AND ActionGeo_Type IN (3, 4, 5)
"""

# 2. 데이터 저장
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("데이터 불러오기 완료")
display(df.head())


Downloading: 100%|██████████|
데이터 불러오기 완료


,SQLDATE,EventCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20230121,192,-9.5,10,-0.840336,4,39.9289,116.388,https://www.shanghaisun.com/news/273404777/tai...
1,20221016,150,-7.2,5,-2.020202,4,39.9289,116.388,https://asia.nikkei.com/Politics/China-s-party...
2,20221016,192,-9.5,2,2.376238,4,25.0478,121.532,https://www.assahifa.com/english/world/chinese...
3,20221016,192,-9.5,6,2.376238,4,25.0478,121.532,https://www.assahifa.com/english/world/chinese...
4,20220612,192,-9.5,2,-5.449190,4,39.9289,116.388,https://www.news24.com/news24/World/News/we-wi...


In [ ]:
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d')

df.to_csv("01_data/raw/gdelt_raw.csv", index=False)
print(f"저장 완료: {len(df)}행")

저장 완료: 11901행


In [2]:
import pandas as pd

df = pd.read_csv("01_data/processed/final_priority.csv")

print(df[['ActionGeo_Lat', 'ActionGeo_Long']].describe())
print(f"\n좌표 결측치: {df[['ActionGeo_Lat', 'ActionGeo_Long']].isnull().sum().to_dict()}")
print(f"\n좌표 샘플:\n{df[['ActionGeo_Lat', 'ActionGeo_Long', 'priority_score']].head(10)}")

       ActionGeo_Lat  ActionGeo_Long
count     177.000000      177.000000
mean       31.823555      115.108647
std         9.472259       23.631831
min       -35.283300      -77.036400
25%        25.047800      116.388000
50%        32.526100      116.388000
75%        39.928900      121.532000
max        55.752200      149.217000

좌표 결측치: {'ActionGeo_Lat': 0, 'ActionGeo_Long': 0}

좌표 샘플:
   ActionGeo_Lat  ActionGeo_Long  priority_score
0        39.9289         116.388        0.851085
1        39.9289         116.388        0.676809
2        24.0000         119.000        0.675123
3        39.9289         116.388        0.634837
4        24.0000         119.000        0.614837
5        39.9289         116.388        0.600000
6        28.7925         117.262        0.549998
7        42.8333         124.633        0.506743
8        24.0000         119.000        0.503507
9        25.0478         121.532        0.498706
